# Traffic Sign Classification — Class Imbalance Study
## 1. Data Loading & EDA

In [1]:
import pandas as pd  # to read CSV files
import numpy as np  # for array operations
from PIL import Image  # to load and resize images
import os  # for file paths
import matplotlib.pyplot as plt  # for plotting

## Loading Data

In [2]:
DATASET_DIR = "dataset"  # base folder where Train/Test/Meta live

train_df = pd.read_csv(os.path.join(DATASET_DIR, "Train.csv"))  # load training metadata (paths + labels)

IMG_SIZE = 32  # resize all images to 32x32 (standard for GTSRB)

images = []  # will hold image arrays
labels = []  # will hold class labels

for idx, row in train_df.iterrows():  # loop through every row in Train.csv
    img_path = os.path.join(DATASET_DIR, row["Path"])  # prepend dataset folder to relative path
    img = Image.open(img_path).convert("RGB")  # open image and force RGB
    img = img.resize((IMG_SIZE, IMG_SIZE))  # resize to fixed size
    images.append(np.array(img))  # convert to array and store
    labels.append(row["ClassId"])  # store corresponding class label

X = np.array(images, dtype="float32") / 255.0  # normalize pixel values to 0-1
y = np.array(labels)  # convert labels list to array

print("X shape:", X.shape)  # sanity check
print("y shape:", y.shape)  # sanity check

X shape: (39209, 32, 32, 3)
y shape: (39209,)


## 2. Train/Validation Split

In [3]:
from sklearn.model_selection import train_test_split  # for stratified splitting

X_train, X_val, y_train, y_val = train_test_split(
    X, y,  # our full dataset
    test_size=0.2,  # 20% data for validation
    stratify=y,  # keep class ratio same in both splits
    random_state=42  # fixed seed for reproducibility
)

print("X_train shape:", X_train.shape)  # check training set size
print("X_val shape:", X_val.shape)  # check validation set size
print("y_train shape:", y_train.shape)  # check training labels size
print("y_val shape:", y_val.shape)  # check validation labels size

X_train shape: (31367, 32, 32, 3)
X_val shape: (7842, 32, 32, 3)
y_train shape: (31367,)
y_val shape: (7842,)


## 3. Baseline CNN Model

In [4]:
import tensorflow as tf  # main deep learning library
from tensorflow.keras import layers, models  # to build CNN layers easily

I0000 00:00:1789407462.888223   25132 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789407464.609697   25132 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789407466.991061   25132 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [5]:
num_classes = 43  # total traffic sign classes

model = models.Sequential([  # simple stack of layers

    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),  # first conv block
    layers.BatchNormalization(),  # stabilize training
    layers.MaxPooling2D((2,2)),  # downsample

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),  # second conv block
    layers.BatchNormalization(),  # stabilize training
    layers.MaxPooling2D((2,2)),  # downsample

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),  # third conv block
    layers.BatchNormalization(),  # stabilize training
    layers.MaxPooling2D((2,2)),  # downsample

    layers.Conv2D(256, (3,3), activation='relu', padding='same'),  # fourth conv block
    layers.BatchNormalization(),  # stabilize training

    layers.GlobalAveragePooling2D(),  # replaces Flatten, reduces overfitting

    layers.Dropout(0.4),  # randomly drop neurons to prevent overfitting
    layers.Dense(128, activation='relu'),  # fully connected layer
    layers.Dropout(0.3),  # another dropout before output

    layers.Dense(num_classes, activation='softmax')  # output layer, one score per class
])

model.summary()  # print architecture summary

/home/mujtaba/Code/Traffic-Signs-Recognition/.venv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1789407467.707848   25132 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 4, 4, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 4, 4, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 43)             │         5,547 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 428,779 (1.64 MB)

 Trainable params: 427,819 (1.63 MB)

 Non-trainable params: 960 (3.75 KB)